# 03b · E2 en la nube: techo de referencia con un LLM comercial grande (ChatGPT / Gemini)

### Para qué

En E2 los LLM locales (7–8 mil millones de parámetros) quedaron 13–31 puntos bajo E1. La pregunta obvia es: **¿y un modelo grande de la nube, como ChatGPT o Gemini?**

- Si un modelo grande llega cerca de E1 **sin entrenamiento**, la brecha de E2 se debe al **tamaño** del modelo local.
- Si tampoco llega, el modelo entrenado (E1) sigue siendo necesario incluso frente a lo mejor disponible.

Se usa **exactamente el mismo prompt, la misma guía, los mismos ejemplos y las mismas métricas** que en el notebook 03; solo cambia el modelo.

### Por qué está permitido aquí (y no en un CESFAM)

El CWLC es **público y anonimizado**, así que enviarlo a una API externa no expone pacientes. Con fichas reales no se podría hacer sin resguardos (Ley 21.719). Por eso este experimento es solo una **referencia**, no la propuesta.

### Cuidados

- **Contaminación:** el corpus es público desde 2022 y un modelo grande pudo haberlo visto al entrenarse. Si rinde muy alto, hay que declararlo.
- **Costo:** se paga por token. Antes de correr todo, la sección 3 mide 5 oraciones y **proyecta el costo**. Nada se corre completo sin confirmarlo.
- **La suscripción no incluye la API:** ChatGPT Plus o Gemini Advanced son productos distintos. La API se activa y paga aparte (platform.openai.com / aistudio.google.com).
- **Las claves nunca van a git:** se guardan en un archivo `.env` en la raíz del proyecto, que git ignora.

### Conexión con el curso

- **U1:** modelos fundacionales comerciales (GPT, Gemini) frente a abiertos (Llama, Qwen).
- **U9:** adaptación por *prompting* e *in-context learning*, sin acceso a los pesos.
- **Ética (CG5):** por qué un modelo de nube no es una opción directa con datos clínicos reales.

In [ ]:
from pathlib import Path
import sys, json, time, random, pickle
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import yaml
import pandas as pd
import matplotlib.pyplot as plt
from datos.cwlc import cargar_split
from evaluacion import metricas
from modelos.e1_encoder import evaluar
from modelos import e2_llm as E2
from reproducibilidad import huella_completa, verificar_huellas_datos, sha256_bytes
from sistema import mantener_despierto, dejar_dormir

cfg_global = yaml.safe_load(open(RAIZ / "configs" / "experimentos.yaml"))
carpeta = RAIZ / "data" / "raw" / "cwlc"
train, dev, test = (cargar_split(carpeta, s) for s in ["train", "dev", "test"])
ok, difs = verificar_huellas_datos(carpeta, "cwlc")
assert ok, f"¡Los datos cambiaron! {difs}"
print("Claves cargadas desde .env:", E2.cargar_claves(RAIZ / ".env") or "ninguna")

## 1. Claves de API

Crear (una sola vez) un archivo llamado `.env` en la carpeta del proyecto, con una línea por proveedor:

```
OPENAI_API_KEY=sk-...
GEMINI_API_KEY=AIza...
```

Git lo ignora, así que nunca se sube. Basta con la clave del proveedor que se vaya a usar; los demás se saltan.

## 2. Configuración

Los modelos y precios están en `configs/experimentos.yaml` (`llm_comercial`). **Hay que confirmar en la página oficial del proveedor** que el nombre del modelo y el precio sigan vigentes: cambian seguido.

- **Estrategias:** zero-shot y *few-shot* dinámico (la mejor de E2). Se omite el *few-shot* fijo para ahorrar costo.
- **`MODO = "rapido"`**: las mismas 100 oraciones del modo rápido de E2. **`"completo"`**: las 990.
- **`CONFIRMAR_GASTO`**: debe ponerse en `True` después de revisar la proyección de costo.

In [ ]:
MODO = "rapido"              # "rapido" (100 oraciones) | "completo" (990)
CONFIRMAR_GASTO = False      # poner True solo después de revisar la proyección de costo

import os
NUBE = [m for m in cfg_global["llm_comercial"] if os.environ.get(E2.PROVEEDORES[m["proveedor"]]["variable"])]
ESTRATEGIAS = ["zero_shot", "few_shot_dinamico"]
oraciones = sorted(random.Random(42).sample(test, 100), key=lambda o: o.id) if MODO == "rapido" else test
dinamico = E2.SelectorDinamico(train, k=cfg_global["k_ejemplos"])
SALIDA = RAIZ / "results" / f"e2nube_{MODO}"
CACHE = RAIZ / "results" / "cache" / f"e2nube_{MODO}"
nombre = lambda m, est: f"{m['proveedor']}_{m['modelo'].replace(':', '-').replace('/', '-')}_{est}"
print("Modelos con clave:", [f"{m['proveedor']}:{m['modelo']}" for m in NUBE] or "NINGUNO (revisar .env)")

## 3. Prueba con 5 oraciones y proyección de costo

Se mandan 5 oraciones por modelo y estrategia (quedan guardadas en la caché, no se pagan dos veces). Con los tokens reales usados se proyecta el costo de la corrida completa.

In [ ]:
filas = []
for m in NUBE:
    llamar = E2.crear_llamador_api(m["proveedor"])
    for est in ESTRATEGIAS:
        cache = E2.Cache(CACHE / f"{nombre(m, est)}.jsonl")
        resp = E2.correr(m["modelo"], oraciones[:5], dinamico if est == "few_shot_dinamico" else None, cache, llamar=llamar, cada=5)
        tok = [r.get("tokens", {}) for r in resp.values()]
        ent = sum(t.get("entrada", 0) for t in tok) / len(tok); sal = sum(t.get("salida", 0) for t in tok) / len(tok)
        costo = len(oraciones) * (ent * m["precio_entrada"] + sal * m["precio_salida"]) / 1e6
        filas.append({"modelo": f"{m['proveedor']}:{m['modelo']}", "estrategia": est, "tokens entrada/oración": round(ent),
                      "tokens salida/oración": round(sal), f"costo proyectado {len(oraciones)} oraciones (USD)": round(costo, 2),
                      "parámetros omitidos": ", ".join(llamar.parametros_omitidos) or "-",
                      "ejemplo de respuesta": next(iter(resp.values()))["respuesta"][:80]})
proyeccion = pd.DataFrame(filas)
display(proyeccion)
print(f"TOTAL proyectado: USD {proyeccion.iloc[:, 4].sum():.2f}")

## 4. Correr

Solo se ejecuta si `CONFIRMAR_GASTO = True`. Igual que en E2: cada respuesta se guarda al llegar, se puede interrumpir y retomar, y cada corrida guarda un JSON con huella, tokens y costo real.

In [ ]:
assert CONFIRMAR_GASTO, "Revisa la proyección de costo y pon CONFIRMAR_GASTO = True para continuar"
despierto = mantener_despierto()
for m in NUBE:
    llamar = E2.crear_llamador_api(m["proveedor"])
    for est in ESTRATEGIAS:
        n = nombre(m, est); ruta = SALIDA / f"{n}.json"
        if ruta.exists():
            print(f"(ya existe) {ruta.name}"); continue
        print(f"\n=== {m['proveedor']}:{m['modelo']} · {est} ===")
        cache = E2.Cache(CACHE / f"{n}.jsonl")
        resp = E2.correr(m["modelo"], oraciones, dinamico if est == "few_shot_dinamico" else None, cache, llamar=llamar, cada=100)
        ev = E2.evaluar_respuestas(resp, oraciones)
        tok_e = sum(r.get("tokens", {}).get("entrada", 0) for r in resp.values())
        tok_s = sum(r.get("tokens", {}).get("salida", 0) for r in resp.values())
        res = {"modelo": f"{m['proveedor']}:{m['modelo']}", "estrategia": est, "tasa_alucinacion": ev["tasa_alucinacion"],
               "json_invalidos": ev["json_invalidos"], "segundos_por_oracion": ev["segundos_por_oracion"],
               "tokens_entrada": tok_e, "tokens_salida": tok_s,
               "costo_usd": (tok_e * m["precio_entrada"] + tok_s * m["precio_salida"]) / 1e6,
               "parametros_omitidos": llamar.parametros_omitidos, "por_tipo": {}}
        for tipo in E2.TIPOS_E2:
            gold = [e for o in oraciones for e in o.entidades if e[3] == tipo]
            res["por_tipo"][tipo] = evaluar(gold, ev["pred"][tipo], oraciones, tipo, train=train)
        config = {**m, "estrategia": est, "k": cfg_global["k_ejemplos"], "temperatura": 0, "guia": sha256_bytes(E2.GUIA.encode()),
                  "modo": MODO, "oraciones": sha256_bytes(json.dumps([o.id for o in oraciones]).encode())}
        huella = huella_completa(config, carpeta, splits={"train": train, "test": test})
        SALIDA.mkdir(parents=True, exist_ok=True)
        ruta.write_text(json.dumps({"config": config, "resultados": res, "huella": huella}, indent=2, ensure_ascii=False, default=str))
        print("  F1 estricto: " + "  ".join(f"{t[:4]}={res['por_tipo'][t]['estricto']['f1']:.3f}" for t in E2.TIPOS_E2)
              + f" · alucinación {ev['tasa_alucinacion']:.1%} · USD {res['costo_usd']:.2f}")
dejar_dormir(despierto)

## 5. Nube vs local vs E1, sobre las mismas oraciones

In [ ]:
ids = {o.id for o in oraciones}
filas = []
for clave in ["espanol_clinico", "chileno_clinico"]:
    for tipo in E2.TIPOS_E2:
        gold = [e for o in oraciones for e in o.entidades if e[3] == tipo]
        f1s = [metricas.contar(gold, [e for e in pickle.loads(p.read_bytes()) if e[0] in ids], "estricto").f1
               for p in (RAIZ / "results" / "cache" / "e1_completo").glob(f"{clave}_{tipo}_s*.pkl")]
        if f1s:
            filas.append({"sistema": f"E1 {clave.split('_')[0]} (entrenado)", "tipo": tipo, "F1": sum(f1s) / len(f1s)})
for carpeta_res, etiqueta in [(RAIZ / "results" / f"e2_{MODO}", "E2 local"), (SALIDA, "E2 nube")]:
    for f in sorted(carpeta_res.glob("*.json")):
        r = json.loads(f.read_text())["resultados"]
        if r["estrategia"] in ESTRATEGIAS:
            for tipo, mt in r["por_tipo"].items():
                filas.append({"sistema": f"{etiqueta} · {r['modelo']} · {r['estrategia']}", "tipo": tipo, "F1": mt["estricto"]["f1"]})
comparacion = pd.DataFrame(filas).pivot(index="sistema", columns="tipo", values="F1").round(3)
display(comparacion)
fig, ax = plt.subplots(figsize=(9, 5))
comparacion.T.plot.bar(ax=ax, rot=0, width=0.85)
ax.set_ylabel("F1 estricto (mismas oraciones)"); ax.set_ylim(0, 1); ax.set_title("E1 vs LLM local vs LLM de nube")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1, 1)); plt.tight_layout(); plt.show()

## 6. Conclusiones (completar)

- ¿El modelo de nube se acerca a E1? ¿Cuánto supera a los locales?
- ¿Mejora también con los ejemplos parecidos, o ya rinde bien sin ejemplos?
- Alucinaciones y JSON inválidos frente a los locales.
- Costo real de la corrida y costo por oración.
- Recordar la posible contaminación (corpus público desde 2022).